In [8]:
import json

# =====================================================
# Paths
# =====================================================

SEMANTIC_PATH = r"E:\Projects\CheckThat 2026\my data\semantic_matches.json"

TRAIN_PATH = r"E:\Projects\CheckThat 2026\task 2\data\train.json"
VAL_PATH = r"E:\Projects\CheckThat 2026\task 2\data\validation.json"
TEST_PATH = r"E:\Projects\CheckThat 2026\task 2\data\test.json"

# =====================================================
# Load files
# =====================================================

with open(SEMANTIC_PATH, "r", encoding="utf-8") as f:
    semantic_matches = json.load(f)

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

# =====================================================
# Extract claims
# =====================================================

semantic_claims = {
    item["new_claim"].strip()
    for item in semantic_matches
    if "new_claim" in item
}

train_claims = {
    item["claim"].strip()
    for item in train_data
}

val_claims = {
    item["claim"].strip()
    for item in val_data
}

test_claims = {
    item["claim"].strip()
    for item in test_data
}

# =====================================================
# Matching
# =====================================================

train_matches = semantic_claims & train_claims
val_matches = semantic_claims & val_claims
test_matches = semantic_claims & test_claims

all_dataset_claims = train_claims | val_claims | test_claims

overall_matches = semantic_claims & all_dataset_claims

unmatched_claims = semantic_claims - all_dataset_claims

# =====================================================
# Statistics
# =====================================================

print("=" * 60)
print("SEMANTIC MATCHES STATISTICS")
print("=" * 60)

print(f"Unique semantic claims      : {len(semantic_claims):,}")
print("using fuzzy logic 95% similarity threshold")
print()

print(f"Matched in train            : {len(train_matches):,}")
print(f"Matched in validation       : {len(val_matches):,}")
print(f"Matched in test             : {len(test_matches):,}")
print()

print(f"Unique matches overall      : {len(overall_matches):,}")
print(f"Unmatched semantic claims   : {len(unmatched_claims):,}")

coverage = (
    len(overall_matches) / len(semantic_claims) * 100
)

print("=" * 60)



SEMANTIC MATCHES STATISTICS
Unique semantic claims      : 6,463
using fuzzy logic 95% similarity threshold

Matched in train            : 6,397
Matched in validation       : 63
Matched in test             : 3

Unique matches overall      : 6,463
Unmatched semantic claims   : 0


# Duplicates in semantic_matches.json

In [14]:
from collections import defaultdict

rows = defaultdict(list)

for idx, item in enumerate(semantic_matches):
    rows[item["new_claim"].strip()].append(idx)

print("Total duplicate claims:", sum(len(indices) for indices in rows.values() if len(indices) > 1)/2)

for claim, indices in rows.items():
    if len(indices) > 1:
        print()
        print("Indices:", indices)
        print("Occurrences:", len(indices))
        print("Claim:", claim[:300])

Total duplicate claims: 6.0

Indices: [1074, 1163]
Occurrences: 2
Claim: Says Rick Scott took $200,000 in campaign contributions from a company that "profited off pollution."

Indices: [5059, 6030]
Occurrences: 2
Claim: "To curb water wastage, the Department of Water and Sanitation has begun its programme of training 15,000 young people as artisans."

Indices: [5791, 5924]
Occurrences: 2
Claim: "There are currently almost one million students who are enrolled in higher education, up from 500,000 in 1994."

Indices: [6463, 6464]
Occurrences: 2
Claim: A 2002 photo accurately depicts 22-year-old Chauntae Davies, an accuser of Jeffrey Epstein, giving former President Bill Clinton a shoulder massage.

Indices: [6465, 6468]
Occurrences: 2
Claim: Prince Andrew said in 2008, “We're not allowed to play Monopoly at home. It gets too vicious."

Indices: [6466, 6467]
Occurrences: 2
Claim: Then-Prince Andrew said in 2008, “We're not allowed to play Monopoly at home. It gets too vicious."


# Statistics on Given Data

In [16]:
import json
from collections import Counter

# =====================================================
# Paths
# =====================================================

TRAIN_PATH = r"E:\Projects\CheckThat 2026\task 2\data\train.json"
VAL_PATH = r"E:\Projects\CheckThat 2026\task 2\data\validation.json"
TEST_PATH = r"E:\Projects\CheckThat 2026\task 2\data\test.json"

# =====================================================
# Helper Function
# =====================================================

def analyze_dataset(path, name):

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    trace_counts = []

    for sample in data:

        # Adjust key name if needed
        count = len(sample.get("Reasoning_traces", []))

        trace_counts.append(count)

    distribution = Counter(trace_counts)

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    print(f"Total Claims           : {len(trace_counts):,}")
    print(f"Min Traces             : {min(trace_counts)}")
    print(f"Max Traces             : {max(trace_counts)}")
    print(f"Average Traces         : {sum(trace_counts)/len(trace_counts):.2f}")

    print("\nDistribution:")
    for k in sorted(distribution):
        print(f"{k:>3} traces : {distribution[k]:,}")

    return trace_counts


# =====================================================
# Analyze Individual Datasets
# =====================================================

train_counts = analyze_dataset(TRAIN_PATH, "Train")
val_counts = analyze_dataset(VAL_PATH, "Validation")
test_counts = analyze_dataset(TEST_PATH, "Test")

# =====================================================
# Combined Statistics
# =====================================================

all_counts = train_counts + val_counts + test_counts

combined_dist = Counter(all_counts)

print("\n" + "=" * 70)
print("COMBINED STATISTICS")
print("=" * 70)

print(f"Total Claims           : {len(all_counts):,}")
print(f"Min Traces             : {min(all_counts)}")
print(f"Max Traces             : {max(all_counts)}")

print("\nDistribution:")
for k in sorted(combined_dist):
    print(f"{k:>3} traces : {combined_dist[k]:,}")


TRAIN
Total Claims           : 6,400
Min Traces             : 15
Max Traces             : 20
Average Traces         : 19.85

Distribution:
 15 traces : 196
 20 traces : 6,204

VALIDATION
Total Claims           : 1,600
Min Traces             : 15
Max Traces             : 15
Average Traces         : 15.00

Distribution:
 15 traces : 1,600

TEST
Total Claims           : 2,558
Min Traces             : 20
Max Traces             : 20
Average Traces         : 20.00

Distribution:
 20 traces : 2,558

COMBINED STATISTICS
Total Claims           : 10,558
Min Traces             : 15
Max Traces             : 20

Distribution:
 15 traces : 1,796
 20 traces : 8,762


# number of traces in semantic_matches.json


In [18]:
import json
from collections import Counter

# =====================================================
# Path
# =====================================================

METADATA_PATH = (
    r"E:\Projects\CheckThat 2026\my data\merged data embeddings"
    r"\new_data_dense_embeddings_metadata.json"
)

# =====================================================
# Load Metadata
# =====================================================

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# =====================================================
# Count Trace Distribution
# =====================================================

trace_counts = Counter()

for item in metadata:
    trace_counts[item["num_traces"]] += 1

# =====================================================
# Print Statistics
# =====================================================

print("=" * 50)
print("TRACE DISTRIBUTION")
print("=" * 50)

total = len(metadata)

for num_traces in sorted(trace_counts):
    count = trace_counts[num_traces]
    percentage = count * 100 / total

    print(
        f"{num_traces:>2} traces : "
        f"{count:>5} claims "
        f"({percentage:.2f}%)"
    )

print("=" * 50)
print(f"Total Claims : {total}")
print("=" * 50)

TRACE DISTRIBUTION
15 traces :   259 claims (4.00%)
20 traces :  6210 claims (96.00%)
Total Claims : 6469


# 2025 data vs 2026 data (in train, val and test splits)

In [19]:
import json
from pathlib import Path

# ============================================================
# FILES
# ============================================================

OLD_2025_FILES = [
    r"E:\Projects\Numerical Fact Checking using SLM\CheckThat 2025\task3\data\1-my dataset\1-train-set-claims.json",
    r"E:\Projects\Numerical Fact Checking using SLM\CheckThat 2025\task3\data\1-my dataset\2-val-set-claims.json",
    r"E:\Projects\Numerical Fact Checking using SLM\CheckThat 2025\task3\data\1-my dataset\3-test-set-claims.json",
]

TRAIN_2026 = r"E:\Projects\CheckThat 2026\task 2\data\train.json"
VAL_2026   = r"E:\Projects\CheckThat 2026\task 2\data\validation.json"
TEST_2026  = r"E:\Projects\CheckThat 2026\task 2\data\test.json"


# ============================================================
# HELPERS
# ============================================================

def normalize(text):
    if text is None:
        return ""

    return (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
        .replace("\r", " ")
    )


def load_claims(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    claims = set()

    if isinstance(data, list):
        for row in data:
            if isinstance(row, dict) and "claim" in row:
                claims.add(normalize(row["claim"]))

    elif isinstance(data, dict):
        for row in data.values():
            if isinstance(row, dict) and "claim" in row:
                claims.add(normalize(row["claim"]))

    return claims


# ============================================================
# LOAD 2025 CLAIMS
# ============================================================

claims_2025 = set()

for file in OLD_2025_FILES:
    claims_2025.update(load_claims(file))

print("=" * 60)
print("2025 DATASET")
print("=" * 60)
print(f"Unique Claims : {len(claims_2025):,}")


# ============================================================
# LOAD 2026 SPLITS
# ============================================================

train_2026 = load_claims(TRAIN_2026)
val_2026   = load_claims(VAL_2026)
test_2026  = load_claims(TEST_2026)

print("\n2026 DATASET")
print("-" * 60)
print(f"Train      : {len(train_2026):,}")
print(f"Validation : {len(val_2026):,}")
print(f"Test       : {len(test_2026):,}")


# ============================================================
# MATCHES
# ============================================================

train_matches = claims_2025.intersection(train_2026)
val_matches   = claims_2025.intersection(val_2026)
test_matches  = claims_2025.intersection(test_2026)

all_matches = (
    train_matches
    | val_matches
    | test_matches
)

# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 60)
print("MATCHING STATISTICS")
print("=" * 60)

print(f"Matched in Train      : {len(train_matches):,}")
print(f"Matched in Validation : {len(val_matches):,}")
print(f"Matched in Test       : {len(test_matches):,}")

print("-" * 60)
print(f"Total Unique Matches  : {len(all_matches):,}")

match_pct = (
    len(all_matches) / len(claims_2025) * 100
    if claims_2025
    else 0
)

print(f"Match Percentage      : {match_pct:.2f}%")
print("=" * 60)

2025 DATASET
Unique Claims : 16,652

2026 DATASET
------------------------------------------------------------
Train      : 6,397
Validation : 1,600
Test       : 1,708

MATCHING STATISTICS
Matched in Train      : 6,397
Matched in Validation : 0
Matched in Test       : 1
------------------------------------------------------------
Total Unique Matches  : 6,398
Match Percentage      : 38.42%


In [26]:
import json
from collections import defaultdict

# ============================================================
# FILES
# ============================================================

SEMANTIC_MATCHES = r"E:\Projects\CheckThat 2026\my data\semantic_matches.json"

TRAIN_2025 = r"E:\Projects\Numerical Fact Checking using SLM\CheckThat 2025\task3\data\1-my dataset\1-train-set-claims.json"
VAL_2025   = r"E:\Projects\Numerical Fact Checking using SLM\CheckThat 2025\task3\data\1-my dataset\2-val-set-claims.json"
TEST_2025  = r"E:\Projects\Numerical Fact Checking using SLM\CheckThat 2025\task3\data\1-my dataset\3-test-set-claims.json"

TRAIN_2026 = r"E:\Projects\CheckThat 2026\task 2\data\train.json"
VAL_2026   = r"E:\Projects\CheckThat 2026\task 2\data\validation.json"
TEST_2026  = r"E:\Projects\CheckThat 2026\task 2\data\test.json"


# ============================================================
# HELPERS
# ============================================================

def normalize(text):
    return (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
        .replace("\r", " ")
    )


def load_claims(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    claims = set()

    if isinstance(data, list):
        for row in data:
            if isinstance(row, dict) and "claim" in row:
                claims.add(normalize(row["claim"]))

    elif isinstance(data, dict):
        for row in data.values():
            if isinstance(row, dict) and "claim" in row:
                claims.add(normalize(row["claim"]))

    return claims


# ============================================================
# LOAD SPLITS
# ============================================================

train25 = load_claims(TRAIN_2025)
val25   = load_claims(VAL_2025)
test25  = load_claims(TEST_2025)

train26 = load_claims(TRAIN_2026)
val26   = load_claims(VAL_2026)
test26  = load_claims(TEST_2026)

# ============================================================
# MATRIX
# ============================================================

matrix = defaultdict(int)

with open(SEMANTIC_MATCHES, "r", encoding="utf-8") as f:
    matches = json.load(f)

for row in matches:

    old_claim = normalize(row["old_claim"])
    new_claim = normalize(row["new_claim"])

    # -------------------------
    # identify 2025 split
    # -------------------------

    if old_claim in train25:
        old_split = "Train25"
    elif old_claim in val25:
        old_split = "Val25"
    elif old_claim in test25:
        old_split = "Test25"
    else:
        continue

    # -------------------------
    # identify 2026 split
    # -------------------------

    if new_claim in train26:
        new_split = "Train26"
    elif new_claim in val26:
        new_split = "Val26"
    elif new_claim in test26:
        new_split = "Test26"
    else:
        continue

    matrix[(old_split, new_split)] += 1


# ============================================================
# PRINT MATRIX
# ============================================================

print("\n")
print("=" * 75)
print("SEMANTIC OVERLAP MATRIX (2025 → 2026)")
print("=" * 75)

print(
    f"{'':12}"
    f"{'Train26':>12}"
    f"{'Val26':>12}"
    f"{'Test26':>12}"
)

rows = ["Train25", "Val25", "Test25"]
cols = ["Train26", "Val26", "Test26"]

for r in rows:

    print(
        f"{r:12}"
        f"{matrix[(r,'Train26')]:12,d}"
        f"{matrix[(r,'Val26')]:12,d}"
        f"{matrix[(r,'Test26')]:12,d}"
    )

print("=" * 75)

total = sum(matrix.values())

print(f"Total Semantic Matches : {total:,}")

print("=" * 75)



SEMANTIC OVERLAP MATRIX (2025 → 2026)
                 Train26       Val26      Test26
Train25            6,400          26           0
Val25                  0           8           0
Test25                 0          29           6
Total Semantic Matches : 6,469
